# Stage 3: Feature Extraction

Demonstrates and caches:
- **MFCCs** (40 coefficients)
- **Mel Spectrograms** (128 bins)
- **Log-Mel Spectrograms**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
import librosa

from data_utils import load_metadata, load_waveform
from features import extract_mfcc, extract_mel, extract_logmel, build_feature_cache, load_feature_cache
from config import SAMPLE_RATE, PLOTS_DIR

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#0d0d1a',
    'text.color': 'white', 'axes.labelcolor': '#aaa',
    'xtick.color': '#aaa', 'ytick.color': '#aaa',
})
print('Ready!')

## 1 · Side-by-Side Feature Comparison

In [ ]:
df = load_metadata()
sample_row = df.iloc[0]
wave = load_waveform(sample_row['filepath'])
print(f'Class: {sample_row["class"]}  |  Wave shape: {wave.shape}')

mfcc   = extract_mfcc(wave)
mel    = extract_mel(wave, log=False)
logmel = extract_logmel(wave)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat, title, cmap in zip(
    axes,
    [mfcc, mel, logmel],
    ['MFCC (40 coeff)', 'Mel Spectrogram', 'Log-Mel Spectrogram'],
    ['coolwarm', 'viridis', 'magma']
):
    im = ax.imshow(feat, aspect='auto', origin='lower', cmap=cmap)
    ax.set_title(title, color='white', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Frames', color='#aaa')
    ax.set_ylabel('Bins / Coefficients', color='#aaa')
    fig.colorbar(im, ax=ax).ax.yaxis.label.set_color('#aaa')

plt.suptitle(f'Feature Comparison — {sample_row["class"]}',
             fontsize=14, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/plots/feature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'MFCC: {mfcc.shape}  Mel: {mel.shape}  LogMel: {logmel.shape}')

## 2 · Delta and Delta-Delta MFCCs

In [ ]:
delta1 = librosa.feature.delta(mfcc)
delta2 = librosa.feature.delta(mfcc, order=2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat, title in zip(axes,
    [mfcc, delta1, delta2],
    ['MFCC', 'Δ MFCC (1st order)', 'ΔΔ MFCC (2nd order)']):
    im = ax.imshow(feat, aspect='auto', origin='lower', cmap='RdBu_r')
    ax.set_title(title, color='white', fontsize=12)
    fig.colorbar(im, ax=ax)

plt.suptitle('MFCC + Delta Features', fontsize=14, color='white', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/plots/mfcc_deltas.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 · Build Feature Cache

> **This cell extracts features for ALL ~4,000 clips. It may take 10–20 minutes.**

In [ ]:
# Extract and cache all three feature types
for ftype in ['mfcc', 'mel', 'logmel']:
    print(f'\n>>> Building {ftype} cache ...')
    build_feature_cache(feature_type=ftype, for_cnn=True)

## 4 · Verify Cache

In [ ]:
X, y, folds = load_feature_cache('logmel')
print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Folds   : {np.unique(folds)}')
print(f'Labels  : {np.unique(y)}')